# Query Classifiers
> Route queries to different retrieval strategies based on their type.

Anchor provides three classifiers: `KeywordClassifier` (rule-based),
`CallbackClassifier` (custom function), and `EmbeddingClassifier`
(semantic similarity). Use them to choose the right retrieval path
for each query.

**Time:** ~8 minutes

## Setup

In [ ]:
from anchor.query.classifiers import (
    KeywordClassifier,
    CallbackClassifier,
    EmbeddingClassifier,
)

## 1. KeywordClassifier
Matches query text against keyword lists for each category. Simple,
fast, and interpretable.

In [ ]:
keyword = KeywordClassifier(
    keywords={
        "technical": ["API", "code", "function", "error", "debug"],
        "general": ["what", "how", "explain", "overview", "introduction"],
        "tutorial": ["step-by-step", "guide", "tutorial", "example"],
    }
)

print(f"Classifier: {type(keyword).__name__}")
print(f"Categories: {list(keyword.keywords.keys())}")

In [ ]:
# Classify several queries
test_queries = [
    "How does the API work?",
    "Explain the architecture overview",
    "Step-by-step guide to building a pipeline",
    "Debug this function error",
    "What is Anchor?",
]

print(f"{'Query':<50} {'Category':>10}")
print("-" * 62)
for q in test_queries:
    result = keyword.classify(q)
    print(f"{q:<50} {result:>10}")

## 2. CallbackClassifier
Delegates classification to a user-defined function. Maximum
flexibility for custom logic.

In [ ]:
def custom_classify(query: str) -> str:
    """Classify based on query characteristics."""
    q = query.lower()
    if any(kw in q for kw in ["api", "code", "function", "class"]):
        return "technical"
    elif any(kw in q for kw in ["compare", "vs", "difference"]):
        return "comparison"
    elif query.endswith("?"):
        return "question"
    else:
        return "general"

callback = CallbackClassifier(callback_fn=custom_classify)

print(f"Classifier: {type(callback).__name__}")

In [ ]:
test_queries = [
    "How does the API handle errors?",
    "Compare RAG vs fine-tuning",
    "Is Anchor production-ready?",
    "Anchor context pipeline overview",
]

print(f"{'Query':<45} {'Category':>12}")
print("-" * 59)
for q in test_queries:
    result = callback.classify(q)
    print(f"{q:<45} {result:>12}")

## 3. EmbeddingClassifier
Uses embedding similarity to classify queries. Each category has
example texts that define its embedding centroid.

In [ ]:
# Mock embedding function (deterministic hash-based)
def embed_fn(text: str) -> list:
    """Produce a fixed-length vector from text."""
    padded = text[:64].ljust(64)
    return [hash(c) % 100 / 100.0 for c in padded]

embedding = EmbeddingClassifier(
    categories={
        "technical": ["API reference", "code example", "function signature"],
        "general": ["overview", "introduction", "getting started"],
        "troubleshooting": ["error message", "bug fix", "debugging"],
    },
    embed_fn=embed_fn,
)

print(f"Classifier: {type(embedding).__name__}")
print(f"Categories: {list(embedding.categories.keys())}")
print(f"Embedding dim: {len(embed_fn('test'))}")

In [ ]:
test_queries = [
    "Show me the API reference",
    "Getting started with Anchor",
    "How to fix this error",
    "Code example for retrieval",
]

print(f"{'Query':<40} {'Category':>16}")
print("-" * 58)
for q in test_queries:
    result = embedding.classify(q)
    print(f"{q:<40} {result:>16}")

## Routing Pattern
Use classification results to route queries to different retrieval
strategies.

In [ ]:
# Define retrieval strategies per category
strategies = {
    "technical": "Use code-aware retriever with AST parsing",
    "general": "Use standard dense retrieval with reranking",
    "comparison": "Use multi-query expansion then merge results",
    "question": "Use HyDE for hypothetical document generation",
    "troubleshooting": "Use step-back prompting for broader context",
}

queries = [
    "How does the API handle rate limiting?",
    "Compare vector and keyword search",
    "Is caching enabled by default?",
    "Anchor framework overview",
]

print("Query routing:\n")
for q in queries:
    category = callback.classify(q)
    strategy = strategies.get(category, "Default retrieval")
    print(f"  Query:    {q}")
    print(f"  Category: {category}")
    print(f"  Strategy: {strategy}")
    print()

## Classifier Comparison

In [ ]:
# Compare all three classifiers on the same queries
comparison_queries = [
    "How does the API work?",
    "Getting started with Anchor",
    "Debug this error in the pipeline",
]

print(f"{'Query':<38} {'Keyword':>12} {'Callback':>12} {'Embedding':>12}")
print("-" * 76)
for q in comparison_queries:
    kw_result = keyword.classify(q)
    cb_result = callback.classify(q)
    em_result = embedding.classify(q)
    print(f"{q:<38} {kw_result:>12} {cb_result:>12} {em_result:>12}")

## Key Takeaways
- **KeywordClassifier**: rule-based, fast, good for well-defined categories
- **CallbackClassifier**: maximum flexibility with custom Python logic
- **EmbeddingClassifier**: semantic matching, adapts to query phrasing
- Use classifiers to route queries to the best retrieval strategy
- Classifiers can be composed: use keyword first, fall back to embedding

**Next:** [Query Pipeline](06_query_pipeline.ipynb)